# 03 — Tách câu / Đoạn cú (斷句) — miễn phí

Hán văn cổ không có dấu câu → **đoạn cú** rồi tách câu, đánh `sentence_id`.

| SEG_ENGINE | Công cụ | Ghi chú |
|-----------|---------|---------|
| `'local'` ⭐ | HuggingFace `raynardj/classical-chinese-punctuation-guwen-biaodian` | Chạy local CPU, free |
| `'gemini'` | Gemini free tier | Cần API key; đoạn cú theo ngữ nghĩa |

Vào: `outputs/<docid>/<docid>_raw.txt` → Ra: `outputs/<docid>/<docid>_seg.tsv`  (`sentence_id \t sentence`).

In [ ]:
# %pip install transformers torch

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent if NB_DIR.name == 'notebooks' else NB_DIR
sys.path.insert(0, str(PROJECT_ROOT / 'src'))   # để import module trong src/

DATA_DIR    = PROJECT_ROOT / 'data'
RAW_IMG_DIR = DATA_DIR / 'raw_images' / 'downloaded_images'   # ảnh gốc theo <docid>/
PREP_DIR    = DATA_DIR / 'preprocessed'
OCR_DIR     = DATA_DIR / 'ocr'
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
for d in [PREP_DIR, OCR_DIR, OUTPUTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def list_works(base):
    """Danh sách 'work id' = tên thư mục con trong base (mỗi tác phẩm/quyển = 1 folder)."""
    if not base.exists():
        return []
    return sorted(p.name for p in base.iterdir() if p.is_dir())

def images_of(base, wid):
    d = base / wid
    return sorted(list(d.glob('*.jpg')) + list(d.glob('*.png')))

print('Project root:', PROJECT_ROOT)
print('Ảnh gốc     :', RAW_IMG_DIR)

In [ ]:
SEG_ENGINE = 'local'   # 'local' hoặc 'gemini'
import re, time
from tqdm.auto import tqdm

def load_raw_text(wid):
    p = OUTPUTS_DIR / wid / f'{wid}_raw.txt'
    if not p.exists():
        return ''
    lines = [ln for ln in p.read_text(encoding='utf-8').splitlines() if not ln.startswith('### ')]
    return re.sub(r'\s+', '', ''.join(lines))   # nối cột thành mạch liền, bỏ trắng

def split_sentences(punctuated):
    parts = re.split(r'(?<=[。！？])', punctuated)
    return [s.strip() for s in parts if s.strip()]

In [ ]:
if SEG_ENGINE == 'local':
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    import torch
    MODEL_ID = 'raynardj/classical-chinese-punctuation-guwen-biaodian'
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    seg_model = AutoModelForTokenClassification.from_pretrained(MODEL_ID).eval()
    id2label = seg_model.config.id2label

    def punctuate(text, max_len=480):
        out = []
        for i in range(0, len(text), max_len):
            chunk = text[i:i+max_len]
            enc = tok(chunk, return_tensors='pt', return_offsets_mapping=True, truncation=True, max_length=512)
            offsets = enc.pop('offset_mapping')[0].tolist()
            with torch.no_grad():
                preds = seg_model(**enc).logits[0].argmax(-1).tolist()
            buf = ''
            for (a, b), pid in zip(offsets, preds):
                if a == b:
                    continue
                buf += chunk[a:b]
                lab = id2label[pid]
                if lab and lab not in ('O', 'S') and len(lab) == 1:
                    buf += lab
            out.append(buf)
        return ''.join(out)
    print('Đã tải model đoạn cú:', MODEL_ID)

In [ ]:
if SEG_ENGINE == 'gemini':
    import os, google.generativeai as genai
    genai.configure(api_key=os.environ.get('GOOGLE_API_KEY') or 'DÁN_API_KEY')
    _m = genai.GenerativeModel('gemini-2.5-flash')
    SEG_PROMPT = ('Thêm dấu câu (。，、？！；：) vào đoạn Hán văn cổ dưới đây để đoạn cú. '
                  'CHỈ thêm dấu câu, KHÔNG thêm/bớt/sửa chữ Hán. Trả về đúng đoạn đã có dấu câu:\n\n')
    def punctuate(text, max_len=1500):
        out = []
        for i in range(0, len(text), max_len):
            try:
                r = _m.generate_content(SEG_PROMPT + text[i:i+max_len]); out.append((r.text or '').strip())
            except Exception as e:
                print('  ! lỗi:', str(e)[:80]); out.append(text[i:i+max_len])
            time.sleep(4.5)
        return ''.join(out)

### Chạy → ghi `_seg.tsv`

In [ ]:
works = list_works(OUTPUTS_DIR)
for wid in works:
    raw = load_raw_text(wid)
    if not raw:
        continue
    sents = split_sentences(punctuate(raw))
    tsv = OUTPUTS_DIR / wid / f'{wid}_seg.tsv'
    with open(tsv, 'w', encoding='utf-8') as f:
        for i, s in enumerate(sents, 1):
            f.write(f'{wid}_{i:06d}\t{s}\n')
    print(f'{wid}: {len(sents)} câu -> {tsv}')

In [ ]:
works = list_works(OUTPUTS_DIR)
if works:
    p = OUTPUTS_DIR / works[0] / f'{works[0]}_seg.tsv'
    if p.exists():
        print('\n'.join(p.read_text(encoding='utf-8').splitlines()[:10]))